# MCP Client Example


Authenticate with Azure AD and interact with the MCP server.

## Authentication with MSAL

In [1]:
from msal import PublicClientApplication
import jwt

# Azure AD app details
PUBLIC_CLIENT_ID = "b746ae15-932a-4529-8830-2c6125af7545"
TENANT_ID = "28042244-bb51-4cd6-8034-7776fa3703e8"
AUTHORITY = f"https://login.microsoftonline.com/{TENANT_ID}"
SCOPE = ["api://fdb29cc1-ddf5-48aa-abd6-393f1858c1c0/Mcp.Access"]

# Create MSAL client
app = PublicClientApplication(client_id=PUBLIC_CLIENT_ID, authority=AUTHORITY)

# Try silent login first
accounts = app.get_accounts()
result = None
if accounts:
    result = app.acquire_token_silent(SCOPE, account=accounts[0])

# Fallback: device code flow
if not result:
    flow = app.initiate_device_flow(scopes=SCOPE)
    print(flow["message"])  # follow link and enter code
    result = app.acquire_token_by_device_flow(flow)

if "access_token" not in result:
    raise Exception(f"Auth failed: {result}")

token = result["access_token"]
print("✅ Token acquired and stored in variable `token`")

To sign in, use a web browser to open the page https://microsoft.com/devicelogin and enter the code D6NMGJVNC to authenticate.
✅ Token acquired and stored in variable `token`


In [2]:
import asyncio
from fastmcp import Client

# Connect to MCP backend
url = "https://gaialz1-brand-dev-euw-mcp-ca.thankfulcoast-319a1fec.westeurope.azurecontainerapps.io/mcp/"
client = Client(url, auth=token)

async def main():
    async with client:
        await client.ping()

        tools = await client.list_tools()
        print("Tools:")
        for t in tools:
            print("-", t)

        resources = await client.list_resources()
        print("\nResources:")
        for r in resources:
            print("-", r)

        prompts = await client.list_prompts()
        print("\nPrompts:")
        for p in prompts:
            print("-", p)

await main()

Tools:
- name='get_zeiss_brand_tone_of_voice' title=None description='Returnss the Zeiss guidelines for tone of voice in texts' inputSchema={'properties': {}, 'type': 'object'} outputSchema={'additionalProperties': True, 'type': 'object'} annotations=None meta={'_fastmcp': {'tags': []}}
- name='get_zeiss_brand_overview' title=None description='Overview of the ZEISS brand, including its name and identity layers.' inputSchema={'properties': {}, 'type': 'object'} outputSchema={'additionalProperties': True, 'type': 'object'} annotations=None meta={'_fastmcp': {'tags': []}}
- name='get_zeiss_brand_experience' title=None description='Returns the ZEISS brand experience principles.' inputSchema={'properties': {}, 'type': 'object'} outputSchema={'additionalProperties': True, 'type': 'object'} annotations=None meta={'_fastmcp': {'tags': []}}

Resources:
- name='resource_tone_of_voice' title=None uri=AnyUrl('resource://zeiss/brand/tone-of-voice.json') description='Returnss the Zeiss guidelines fo

## Tools

In [4]:
async with client:
    result = await client.call_tool("get_zeiss_brand_experience", {})
    print(result.content[0].text)

{"context":{"author":"Sven Spöde","department":"CBC-BSD","created":"2025-07-04T16:32:00Z","type":"brand_experience_principles","tags":["ZEISS","Brand Identity","Experience Principles"]},"metadata":{"title":"Brand Experience Principles","description":"Describes the five brand experience principles: Leading, Inspiring, Precise, Responsible, and Open, and explains how they are defined."},"blob":{"definition":{"explanation":"The attributes of the brand experience principles are described in the context of:","contexts":[{"name":"Our audiences","description":"The intended relationship with ZEISS. We want people to feel…"},{"name":"Our touchpoints","description":"How we bring it to life at ZEISS. A ZEISS Experience…"},{"name":"Our brand personality","description":"How we collectively show up as ZEISS. We are…"}]},"principles":[{"name":"Leading","contexts":{"ourAudiences":"We want people to feel equipped and self-assured, ahead of the curve.","ourTouchpoints":"A ZEISS Experience makes the futu

## Prompts

In [ ]:
async with client:
    # List available prompts
    prompts = await client.list_prompts()
    
    # Get a rendered prompt
    messages = await client.get_prompt("prompt_ai_charachters",{})
    print(messages.messages)

## Resources

In [ ]:
async with client:
    # List available resources
    resources = await client.list_resources()
    
    # Read a resource
    content = await client.read_resource("resource://zeiss/brand/tone-of-voice.json")
    print(content[0].text)